In [2]:
import pandas as pd
import numpy as np

In [3]:
ms = pd.read_csv('/content/marketing_spend.csv' , thousands=',')
o = pd.read_csv('/content/orders.csv' , thousands = ',')

In [4]:
# Verify that order-level costs add up correctly (product cost + shipping + fees = total costs).

(o.loc[: , ['product_cost','shipping_cost','platform_fee','transaction_fee']].sum(axis = 1).round(2) == o.loc[:,'total_costs'].round(2)).all()

np.True_

In [5]:
# Group orders by product category.
# Calculate total revenue, total costs, total profit, and profit margin for each.
# Identify the top and bottom performers.

grouped_order = o.groupby('primary_category' , as_index= False)[['total_costs', 'profit', 'net_revenue']].sum()

grouped_order['profit_margin'] = (grouped_order['profit'] / grouped_order['net_revenue'] * 100).round(2)

grouped_order


,primary_category,total_costs,profit,net_revenue,profit_margin
0,Beauty,15079.19,3174.88,18254.07,17.39
1,Books,16597.58,2249.79,18847.37,11.94
2,Clothing,25110.65,6272.48,31383.13,19.99
3,Electronics,30912.38,13973.46,44885.84,31.13
4,Food & Beverage,23070.92,7591.77,30662.69,24.76
5,Home & Kitchen,19671.19,6688.57,26359.76,25.37
6,Sports,24733.20,7596.63,32329.83,23.50
7,Toys,24809.43,8786.25,33595.68,26.15


In [6]:
# Group by sales channel.
# Compare average order value, average profit, and return rate across channels.


o['is_returned'] = o['returned'].apply(lambda x : 0 if x == 'No' else 1)

grouped_order = o.groupby('channel' , as_index= False).agg({'gross_revenue' : 'mean', 'profit' : 'mean','is_returned' : lambda x : x.mean() * 100 })

grouped_order


,channel,gross_revenue,profit,is_returned
0,Marketplace,137.545203,15.397446,6.443914
1,Mobile App,140.451511,36.321392,7.300509
2,Social Commerce,134.246751,17.109340,9.137056
3,Website,139.830365,31.595547,7.044025


In [7]:
# Factor in platform fees for Marketplace and Social Commerce.

grouped_order = o.groupby('channel' , as_index= False).agg({'platform_fee' : 'mean'})
grouped_order.loc[0:2:2 ,:]


,channel,platform_fee
0,Marketplace,18.973007
2,Social Commerce,9.872741


In [8]:
# Calculate ROAS, cost per acquisition, and cost per click by platform.

# Identify which platforms are underperforming.

grouped_marketing_spend = ms.groupby('platform', as_index = False).agg({'roas' : 'mean' , 'cpa' : 'mean' , 'cpc' : 'mean'})

grouped_marketing_spend


,platform,roas,cpa,cpc
0,Email Marketing,5.405417,26.014167,1.092500
1,Facebook Ads,11.254583,8.383750,0.361250
2,Google Ads,13.688750,6.483333,0.285000
3,Influencer,23.446667,4.798750,0.185000
4,Instagram Ads,16.990000,5.545417,0.225833
5,TikTok Ads,24.435417,3.930417,0.160417


In [9]:
# What is the average profit margin by product category? Which categories are the most and least profitable,
# and what is driving the difference (product cost, shipping, returns, or discounts)?

grouped_order = o.groupby('primary_category', as_index = False).agg({ 'order_id' : 'count' , 'is_returned' : 'sum' , 'product_cost' : 'mean' , 'shipping_cost':'mean' , 'discount_amount' : 'mean' , 'net_revenue' : 'sum' , 'total_costs' : 'sum' , 'profit' : 'sum'})

grouped_order = grouped_order.assign(
                              profit_margin = round(grouped_order['profit'] / grouped_order['net_revenue'] * 100,2),
                              return_rate = round(grouped_order['is_returned'] / grouped_order['order_id'] * 100 , 2))

grouped_order



,primary_category,order_id,is_returned,product_cost,shipping_cost,discount_amount,net_revenue,total_costs,profit,profit_margin,return_rate
0,Beauty,205,12,41.905317,25.234049,8.544341,18254.07,15079.19,3174.88,17.39,5.85
1,Books,239,20,37.476444,26.204184,6.107741,18847.37,16597.58,2249.79,11.94,8.37
2,Clothing,293,24,52.373481,25.408191,9.976997,31383.13,25110.65,6272.48,19.99,8.19
3,Electronics,267,23,75.247416,26.740824,14.653109,44885.84,30912.38,13973.46,31.13,8.61
4,Food & Beverage,247,14,57.890972,26.080000,10.976802,30662.69,23070.92,7591.77,24.76,5.67
5,Home & Kitchen,200,12,61.743350,24.944200,10.897850,26359.76,19671.19,6688.57,25.37,6.00
6,Sports,292,21,53.738425,23.680548,10.667740,32329.83,24733.20,7596.63,23.50,7.19
7,Toys,257,18,60.741790,26.227082,11.733502,33595.68,24809.43,8786.25,26.15,7.00


In [10]:
# How does profitability differ across sales channels (Website, Mobile App, Marketplace, Social Commerce)?
# Which channel has the best and worst profit per order after accounting for platform fees?

grouped_order = o.groupby('channel').agg(total_order = ('order_id' , 'count'), total_profit = ('profit' , 'sum'))

grouped_order = grouped_order.assign(profit_per_order = round(grouped_order['total_profit'] / grouped_order['total_order'],2))

grouped_order['profit_per_order']


,profit_per_order
channel,
Marketplace,15.40
Mobile App,36.32
Social Commerce,17.11
Website,31.60


In [11]:
# What is the return rate by channel? Estimate how much total revenue was lost to returns over the analysis period.

condition = o['is_returned'] == 1

grouped_order = o.loc[condition].groupby('channel', as_index = False).agg(total_lost_revenue = ('gross_revenue' , 'sum'))

return_rate = o.groupby('channel', as_index = False).agg({'is_returned' : 'sum' , 'order_id' : 'count'})

return_rate['return_rate'] = round(return_rate.loc[: , 'is_returned'] / return_rate['order_id'] * 100,2)

grouped_order.merge(return_rate[['channel' , 'return_rate']] , on= 'channel' , how = "left")

,channel,total_lost_revenue,return_rate
0,Marketplace,3831.02,6.44
1,Mobile App,5713.24,7.30
2,Social Commerce,2662.03,9.14
3,Website,10000.47,7.04


In [12]:
# What is the return rate by category? Estimate how much total revenue was lost to returns over the analysis period.

condition = o['is_returned'] == 1

grouped_order = o.loc[condition].groupby('primary_category', as_index = False).agg(total_lost_revenue = ('gross_revenue' , 'sum'))

return_rate = o.groupby('primary_category', as_index = False).agg({'is_returned' : 'sum' , 'order_id' : 'count'})

return_rate['return_rate'] = round(return_rate.loc[: , 'is_returned'] / return_rate['order_id'] * 100,2)

grouped_order.merge(return_rate[['primary_category' , 'return_rate']] , on= 'primary_category' , how = "left")

,primary_category,total_lost_revenue,return_rate
0,Beauty,923.97,5.85
1,Books,2353.74,8.37
2,Clothing,3448.82,8.19
3,Electronics,4305.35,8.61
4,Food & Beverage,3713.08,5.67
5,Home & Kitchen,2548.83,6.00
6,Sports,2159.00,7.19
7,Toys,2753.97,7.00


In [13]:
# Analyze the marketing spend data: Which advertising platform delivers the best ROAS (Return on Ad Spend)?
# Are there any platforms where the company is spending money but not getting a positive return?

grouped_market_spend = round(ms.groupby('platform')['roas'].mean(), 2)

grouped_market_spend


,roas
platform,
Email Marketing,5.41
Facebook Ads,11.25
Google Ads,13.69
Influencer,23.45
Instagram Ads,16.99
TikTok Ads,24.44
